# Phase 13 — Unified Final Test Evaluation

This notebook reviews the saved official IMDb test evaluation for Logistic Regression, TextCNN, vanilla RNN, LSTM, and GRU. The reusable package runner loads the frozen artifacts, predicts all test rows, computes metrics, benchmarks inference, and writes a new timestamped directory. It does not train models or fit preprocessing. The Phase 7 Logistic Regression and TextCNN figures are historical results, which this phase reproduces.

Set `RUN_EVALUATION = True` only when intentionally running the complete Phase 13 evaluation again. By default, the notebook inspects the already completed Phase 13 output.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from deeptext.data.dataset import load_imdb_split
from deeptext.evaluation.comparison import evaluate_comparison_model
from deeptext.experiments.final_evaluation import _validate_test_records
from deeptext.inference.predictor import (
    load_recurrent_predictor,
    load_textcnn_predictor,
)
from deeptext.models.baseline import load_baseline_artifacts

RUN_EVALUATION = False
if RUN_EVALUATION:
    from deeptext.experiments.phase_13_final_evaluation import run_phase13_evaluation

    run_phase13_evaluation()

OUTPUT_ROOT = Path("outputs/final_evaluation")
evaluation_files = sorted(
    OUTPUT_ROOT.glob("imdb-phase13-final-evaluation-*/metadata.json")
)
if not evaluation_files:
    raise FileNotFoundError(
        "Run the Phase 13 evaluation command before reviewing results."
    )
evaluation_dir = evaluation_files[-1].parent
metadata = json.loads((evaluation_dir / "metadata.json").read_text(encoding="utf-8"))
comparison = json.loads(
    (evaluation_dir / "comparison.json").read_text(encoding="utf-8")
)
metrics_data = json.loads((evaluation_dir / "metrics.json").read_text(encoding="utf-8"))
benchmark_data = json.loads(
    (evaluation_dir / "benchmark.json").read_text(encoding="utf-8")
)
print("Evaluation directory:", evaluation_dir)
print("Official test reviews:", metadata["sample_count"])

## 1. Load and validate the official test split

This uses the existing dataset loader and validates source IDs, class labels, counts, and loader order. No review text is displayed in this step.

In [ ]:
DATASET_DIR = Path("datasets/aclImdb")
test_records = load_imdb_split(DATASET_DIR, "test")
test_validation = _validate_test_records(test_records, DATASET_DIR)
assert len(test_records) == metadata["sample_count"] == 25_000
assert (
    test_validation["review_id_order_sha256"] == metadata["test_review_id_order_sha256"]
)
print(test_validation["class_distribution"])
print("Test row order matches saved evaluation:", True)

## 2. Load the five saved model artifacts

These are the same inference loaders used by the package evaluation. Models are loaded on CPU here for inspection only; this cell does not train or predict.

In [ ]:
artifact_paths = {
    key: Path(value) for key, value in metadata["source_artifacts"].items()
}
baseline_model, baseline_metadata = load_baseline_artifacts(
    artifact_paths["logistic_regression"]
)
loaded_models = {
    "textcnn": load_textcnn_predictor(artifact_paths["textcnn"], "cpu"),
    "rnn": load_recurrent_predictor(artifact_paths["rnn"], "cpu"),
    "lstm": load_recurrent_predictor(artifact_paths["lstm"], "cpu"),
    "gru": load_recurrent_predictor(artifact_paths["gru"], "cpu"),
}
print("Loaded baseline:", baseline_metadata["model_name"])
print(
    "Loaded neural models:",
    [predictor.metadata["model_name"] for predictor in loaded_models.values()],
)

## 3. Review saved predictions and recompute metrics

The CSVs contain one row per official test review in the original loader order. Predictions were generated in batches by the shared inference interfaces and saved without duplicating full review text.

In [ ]:
labels = test_records["label"].to_numpy(dtype=np.int64)
test_ids = test_records["review_id"].tolist()
prediction_frames = {}
recomputed_metrics = {}
for model_key in comparison["models"]:
    frame = pd.read_csv(evaluation_dir / "predictions" / f"{model_key}.csv")
    assert len(frame) == len(test_records)
    assert frame["review_id"].tolist() == test_ids
    assert np.array_equal(frame["true_label"].to_numpy(dtype=np.int64), labels)
    prediction_frames[model_key] = frame
    values = evaluate_comparison_model(
        labels,
        frame["predicted_label"].to_numpy(),
        frame["positive_probability"].to_numpy(),
    )
    recomputed_metrics[model_key] = values
    for name in ("accuracy", "precision", "recall", "f1"):
        assert np.isclose(
            values[name], metrics_data["models"][model_key][name], atol=1e-12, rtol=0
        )
metric_table = pd.DataFrame(
    {
        key: {
            name: values[name]
            for name in (
                "accuracy",
                "precision",
                "recall",
                "f1",
                "false_positive",
                "false_negative",
            )
        }
        for key, values in metrics_data["models"].items()
    }
).T
print(metric_table.to_string(float_format=lambda value: f"{value:.4f}"))

## 4. Classification reports and confusion matrices

Confusion-matrix rows are actual labels and columns are predicted labels in `[[TN, FP], [FN, TP]]` order. Per-class reports are included in `metrics.json`.

In [ ]:
for model_key in comparison["models"]:
    matrix = json.loads(
        (evaluation_dir / "confusion_matrices" / f"{model_key}.json").read_text(
            encoding="utf-8"
        )
    )["matrix"]
    report = metrics_data["classification_reports"][model_key]
    print(model_key, "confusion matrix:", matrix)
    print("  negative:", report["negative"])
    print("  positive:", report["positive"])

## 5. Error analysis and model agreement

Review text is stored only for a deterministic, capped sample. Confidence is the larger of the two model probabilities; it is not a guarantee of correctness. Counts cover all 25,000 rows.

In [ ]:
error_summary = json.loads(
    (evaluation_dir / "error_analysis" / "summary.json").read_text(encoding="utf-8")
)
examples = pd.read_csv(evaluation_dir / "error_analysis" / "selected_examples.csv")
print("Error and agreement counts:", error_summary["counts"])
print(
    examples[["review_id", "true_sentiment", "analysis_tags", "review_text"]]
    .head(8)
    .to_string(index=False)
)
agreement = pd.DataFrame(comparison["pairwise_prediction_agreement"]).T
print("Pairwise prediction agreement:\n", agreement.to_string())
print("Joint counts:", comparison["joint_prediction_counts"])

## 6. Inference benchmarks and saved visualizations

Full-dataset timings cover the entire 25,000-row split. Neural CPU and CUDA runs use the same batch size, with CUDA synchronization. Single-review latency uses the same class-balanced 100-review sample and includes raw-text preprocessing.

In [ ]:
benchmark_rows = []
for model_key, devices in benchmark_data.items():
    for device_key, values in devices.items():
        benchmark_rows.append(
            {
                "model": model_key,
                "device": values["device"],
                "full_test_seconds": values["total_test_inference_seconds"],
                "full_test_reviews_per_second": values[
                    "full_dataset_reviews_per_second"
                ],
                "single_review_mean_ms": values["single_review_latency"][
                    "mean_latency_ms"
                ],
                "batch_size": values["batch_size"],
            }
        )
benchmark_table = pd.DataFrame(benchmark_rows)
print(benchmark_table.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

for plot_path in metadata["generated_plots"]:
    image = plt.imread(evaluation_dir / plot_path)
    plt.figure(figsize=(9, 4.5))
    plt.imshow(image)
    plt.axis("off")
    plt.title(Path(plot_path).name.replace("_", " "))
    plt.show()

## 7. Phase 7 consistency and final summary

Phase 7 already evaluated Logistic Regression and TextCNN on this official test set. Phase 13 reproduces those same frozen-model predictions and metrics; the three recurrent models are evaluated on the official test set here for the first time. The official test set must not be used for training, tuning, preprocessing fit, or model selection.

In [ ]:
consistency = pd.DataFrame(comparison["phase7_consistency"]).T
print(
    consistency[
        [
            "metrics_match_within_1e-12",
            "prediction_label_differences",
            "metric_differences",
        ]
    ].to_string()
)
print("All five models evaluated:", list(metrics_data["models"]))
print("All source model artifacts unchanged:", metadata["artifacts_unchanged"])
print("Official test reviews evaluated:", metadata["sample_count"])
print(
    "New test-based training or model selection:",
    metadata["model_training_started"] or metadata["test_based_model_selection"],
)